# Model Improvement -- Weighted Loss + Extended Fine-Tuning

**Purpose**: document the iterative improvement of EfficientNet-B0 on `metal_nut`.

## Motivation

Version 1 (Commit 6) used unweighted `CrossEntropyLoss` with 10+10 epochs:
- F1 = 0.634 (t=0.5) | Recall = 46% -- only 13/28 defects caught
- Root cause: class imbalance (242 good vs 93 defective, ratio 2.6:1)
- CrossEntropyLoss treats all samples equally => model learns to predict 'good' by default

**Techniques applied**:
1. **Weighted CrossEntropyLoss** -- penalises false negatives proportionally to class imbalance
2. **Extended Phase 2** -- 10 to 20 epochs at lr=1e-5

Both are standard practice for imbalanced binary classification in industrial inspection,
where a missed defect (false negative) is far more costly than a false alarm (false positive).

---

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix

from src.dataset import MVTecTorchDataset
from src.models.deep import DeepClassifier, get_transforms

DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_ROOT    = Path('../data/mvtec_ad')
RANDOM_STATE = 42
BATCH_SIZE   = 32
CATEGORY     = 'metal_nut'
CKPT_V1      = Path('../outputs/checkpoints/efficientnet_metal_nut.pt')
CKPT_V2      = Path('../outputs/checkpoints/efficientnet_metal_nut_v2.pt')

print(f'Device: {DEVICE}')
print(f'V1 checkpoint exists: {CKPT_V1.exists()}')

## 1. Baseline -- V1 metrics (Commit 6 checkpoint)

In [ ]:
def get_test_loader():
    paths, labels = MVTecTorchDataset.collect_paths(DATA_ROOT / CATEGORY)
    _, test_p, _, test_l = train_test_split(
        paths, labels, test_size=0.30, random_state=RANDOM_STATE, stratify=labels
    )
    ds = MVTecTorchDataset(test_p, test_l, transform=get_transforms(train=False))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

def evaluate_ckpt(ckpt_path, threshold=0.5):
    model = DeepClassifier(num_classes=2)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval().to(DEVICE)
    loader = get_test_loader()
    y_pred, y_proba, y_true = [], [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            out = torch.softmax(model(imgs.to(DEVICE)), dim=1).cpu()
            y_proba.extend(out[:, 1].numpy())
            y_pred.extend((out[:, 1] >= threshold).int().numpy())
            y_true.extend(lbls.numpy())
    yt, yp, ypr = map(np.array, [y_true, y_pred, y_proba])
    return {
        'f1':        f1_score(yt, yp, zero_division=0),
        'recall':    recall_score(yt, yp, zero_division=0),
        'precision': precision_score(yt, yp, zero_division=0),
        'accuracy':  accuracy_score(yt, yp),
        'y_true': yt, 'y_pred': yp, 'y_proba': ypr
    }

v1_05 = evaluate_ckpt(CKPT_V1, 0.5)
v1_03 = evaluate_ckpt(CKPT_V1, 0.3)
print('V1 @ t=0.5  F1={:.3f}  Recall={:.3f}  Precision={:.3f}'.format(v1_05['f1'], v1_05['recall'], v1_05['precision']))
print('V1 @ t=0.3  F1={:.3f}  Recall={:.3f}  Precision={:.3f}'.format(v1_03['f1'], v1_03['recall'], v1_03['precision']))

## 2. Diagnosis -- Class imbalance

In [ ]:
paths, labels = MVTecTorchDataset.collect_paths(DATA_ROOT / CATEGORY)
labels_arr = np.array(labels)
n_good   = (labels_arr == 0).sum()
n_defect = (labels_arr == 1).sum()
ratio    = n_good / n_defect

print(f'Total: {len(labels_arr)}  Good: {n_good}  Defective: {n_defect}  Ratio: {ratio:.2f}:1')
print(f'Weighted loss weight for defective class: {ratio:.2f}')
print(f'Each defective sample counts {ratio:.1f}x more in gradient updates')

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['Good (0)', 'Defective (1)'], [n_good, n_defect], color=['steelblue', 'tomato'], alpha=0.85)
ax.set_title('Class Distribution -- metal_nut (all splits combined)')
ax.set_ylabel('Count')
for bar, val in zip(ax.patches, [n_good, n_defect]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, str(val), ha='center', fontsize=12)
plt.tight_layout()
plt.show()

## 3. V2 Training -- Weighted Loss + Extended Phase 2

| Parameter | V1 | V2 | Rationale |
|---|---|---|---|
| Loss | CrossEntropyLoss() | CrossEntropyLoss(weight=[1.0, 2.6]) | Compensate 2.6:1 imbalance |
| Phase 1 epochs | 10 | 10 | Head converges quickly -- unchanged |
| Phase 2 epochs | 10 | 20 | Backbone needs more steps for industrial textures |
| Learning rates | 1e-3 / 1e-5 | 1e-3 / 1e-5 | Unchanged |

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total = 0.0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), lbls)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(imgs)
    return total / len(loader.dataset)

# Build train loader
paths, labels = MVTecTorchDataset.collect_paths(DATA_ROOT / CATEGORY)
train_p, _, train_l, _ = train_test_split(
    paths, labels, test_size=0.30, random_state=RANDOM_STATE, stratify=labels
)
train_ds     = MVTecTorchDataset(train_p, train_l, transform=get_transforms(train=True))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

n_good_tr   = sum(1 for l in train_l if l == 0)
n_defect_tr = sum(1 for l in train_l if l == 1)
w_defect    = round(n_good_tr / n_defect_tr, 2)
weights     = torch.tensor([1.0, w_defect]).to(DEVICE)
criterion   = nn.CrossEntropyLoss(weight=weights)
print(f'Train: {n_good_tr} good, {n_defect_tr} defective')
print(f'Loss weights: good=1.00  defective={w_defect}')

model_v2 = DeepClassifier(num_classes=2, freeze_backbone=True).to(DEVICE)
history  = {'phase': [], 'epoch': [], 'loss': []}
PHASE1_EP, PHASE2_EP = 10, 20

# Phase 1 -- head only
opt1 = optim.Adam(filter(lambda p: p.requires_grad, model_v2.parameters()), lr=1e-3)
print(f'\nPhase 1 ({PHASE1_EP} epochs, head only) ...')
for ep in range(1, PHASE1_EP + 1):
    loss = train_one_epoch(model_v2, train_loader, opt1, criterion, DEVICE)
    history['phase'].append(1); history['epoch'].append(ep); history['loss'].append(loss)
    if ep % 5 == 0: print(f'  ep {ep:2d}/{PHASE1_EP}  loss={loss:.4f}')

# Phase 2 -- full network
model_v2.unfreeze_backbone()
opt2 = optim.Adam(model_v2.parameters(), lr=1e-5)
print(f'Phase 2 ({PHASE2_EP} epochs, full network) ...')
for ep in range(1, PHASE2_EP + 1):
    loss = train_one_epoch(model_v2, train_loader, opt2, criterion, DEVICE)
    history['phase'].append(2); history['epoch'].append(ep); history['loss'].append(loss)
    if ep % 5 == 0: print(f'  ep {ep:2d}/{PHASE2_EP}  loss={loss:.4f}')

torch.save(model_v2.state_dict(), CKPT_V2)
print(f'\nCheckpoint saved: {CKPT_V2}')

## 4. Training curves -- loss over 30 epochs

In [ ]:
import os
df_h  = pd.DataFrame(history)
p1    = df_h[df_h['phase'] == 1]
p2    = df_h[df_h['phase'] == 2]
ep_p1 = list(range(len(p1)))
ep_p2 = list(range(len(p1), len(p1) + len(p2)))

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(ep_p1, p1['loss'].values, color='steelblue', lw=2, label='Phase 1 (head only, lr=1e-3)')
ax.plot(ep_p2, p2['loss'].values, color='tomato',    lw=2, label='Phase 2 (full network, lr=1e-5)')
ax.axvline(len(p1), color='gray', ls='--', alpha=0.6, label='Phase boundary')
ax.axvspan(0,       len(p1),           alpha=0.07, color='steelblue')
ax.axvspan(len(p1), len(p1)+len(p2),   alpha=0.07, color='tomato')
ax.set_xlabel('Epoch')
ax.set_ylabel('Weighted Cross-Entropy Loss')
ax.set_title('V2 Training Curves -- Weighted Loss, 10+20 Epochs')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
os.makedirs('../outputs/results', exist_ok=True)
plt.savefig('../outputs/results/training_curves_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/results/training_curves_v2.png')

## 5. V1 vs V2 full comparison

In [ ]:
v2_05 = evaluate_ckpt(CKPT_V2, 0.5)
v2_03 = evaluate_ckpt(CKPT_V2, 0.3)

rows = [
    {'Version': 'V1 -- baseline (t=0.5)',         'F1': v1_05['f1'], 'Recall': v1_05['recall'], 'Precision': v1_05['precision'], 'Accuracy': v1_05['accuracy']},
    {'Version': 'V1 -- threshold tuned (t=0.3)',  'F1': v1_03['f1'], 'Recall': v1_03['recall'], 'Precision': v1_03['precision'], 'Accuracy': v1_03['accuracy']},
    {'Version': 'V2 -- weighted loss (t=0.5)',    'F1': v2_05['f1'], 'Recall': v2_05['recall'], 'Precision': v2_05['precision'], 'Accuracy': v2_05['accuracy']},
    {'Version': 'V2 -- weighted + tuned (t=0.3)', 'F1': v2_03['f1'], 'Recall': v2_03['recall'], 'Precision': v2_03['precision'], 'Accuracy': v2_03['accuracy']},
]
df = pd.DataFrame(rows)
for col in ['F1','Recall','Precision','Accuracy']:
    df[col] = df[col].map('{:.3f}'.format)
print('=== EfficientNet-B0 metal_nut -- V1 vs V2 ===')
print(df.to_string(index=False))

d_f1  = v2_03['f1']     - v1_03['f1']
d_rec = v2_03['recall'] - v1_03['recall']
sign  = lambda x: '+' if x >= 0 else ''
print(f'\nDelta (V2 t=0.3 vs V1 t=0.3):')
print(f'  F1     {v1_03["f1"]:.3f} -> {v2_03["f1"]:.3f}  ({sign(d_f1)}{d_f1:.3f})')
print(f'  Recall {v1_03["recall"]:.3f} -> {v2_03["recall"]:.3f}  ({sign(d_rec)}{d_rec:.3f})')

## 6. Confusion matrices -- fewer missed defects is the goal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
pairs = [
    ('V1 baseline (t=0.5)', v1_05['y_true'], v1_05['y_pred']),
    ('V2 weighted (t=0.3)', v2_03['y_true'], v2_03['y_pred']),
]
for ax, (title, yt, yp) in zip(axes, pairs):
    cm = confusion_matrix(yt, yp)
    im = ax.imshow(cm, cmap='Blues')
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0,1]); ax.set_xticklabels(['Good','Defective'])
    ax.set_yticks([0,1]); ax.set_yticklabels(['Good','Defective'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(title)
    for r in range(2):
        for c in range(2):
            ax.text(c, r, str(cm[r,c]), ha='center', va='center',
                    color='white' if cm[r,c] > cm.max()/2 else 'black', fontsize=14)
    tn,fp,fn,tp = cm.ravel()
    ax.set_xlabel(f'Predicted  |  Missed defects (FN): {fn}')
plt.suptitle('V1 vs V2 -- impact of weighted loss on false negatives', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/results/confusion_v1_v2.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Promote V2 checkpoint if better

In [ ]:
import shutil

v1_best = max(v1_05['f1'], v1_03['f1'])
v2_best = max(v2_05['f1'], v2_03['f1'])
print(f'Best V1 F1: {v1_best:.3f}')
print(f'Best V2 F1: {v2_best:.3f}')

if v2_best >= v1_best:
    shutil.copy2(CKPT_V2, CKPT_V1)
    print(f'V2 promoted to {CKPT_V1}')
    print('Demo and all notebooks now use the improved model.')
else:
    print('V1 still better on this split -- keeping original checkpoint.')
    print('Try increasing PHASE2_EP or the defective weight.')

## 8. Summary

| Technique | Effect | Why it works |
|---|---|---|
| `CrossEntropyLoss(weight=[1.0, 2.6])` | Penalises false negatives more | Each defective sample counts 2.6x -- compensates majority-class bias |
| Phase 2: 10 -> 20 epochs | More fine-tuning of backbone | ImageNet features need more gradient steps to adapt to industrial textures |

**Key exam insight**: the improvement is not about a smarter architecture -- it is about correctly
formulating the learning problem. Class imbalance is a data problem, not a model problem.
Weighted loss is mathematically equivalent to oversampling the minority class but more numerically stable.

This notebook documents the full before/after story: V1 underperformed because the loss function
did not reflect the asymmetric cost of errors in industrial inspection.